## Objective

- Scrape the reviews of the film *Winny* from the Filmarks movie review website using BeautifulSoup, collect the ratings and review text in a DataFrame, and save the data to a CSV file. Then remove noisy characters such as symbols and emojis from the review text using `str.replace`. This creates a preprocessed dataset (`review_clean.csv`) for subsequent annotation (assigning ground-truth labels) and sentiment analysis with BERT.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/My Drive/test/'   # Modify the path or directory to match your environment

Mounted at /content/drive
/content/drive/My Drive/Deep learning/sc_nlp


In [ ]:
# Scrape data from Filmarks
# Target film: "Winny"
from bs4 import BeautifulSoup
from urllib import request

# Open the specified URL and retrieve its contents
# in the same way as reading a file
url = 'https://filmarks.com/movies/84467/'
response = request.urlopen(url)

# Retrieve the page source
soup = BeautifulSoup(response)
response.close()

# Display the page source
print(soup)

<!DOCTYPE html>
<html lang="ja"><head><meta charset="utf-8"/><title>Winny - 映画情報・レビュー・評価・あらすじ・動画配信 | Filmarks映画</title><meta content="株式会社つみき" name="author"/><meta content="Winnyの映画情報。7734件のレビュー(口コミ・感想・評価)、内容・ネタバレ、あらすじ、予告編・予告動画、公開映画館情報、公開スケジュール、監督・出演者の関連映画情報、Winnyの動画を配信している動画配信サービスの情報。松本優作監督、東出昌大出演。2002 年、開発者・⾦⼦勇（東出昌⼤）は、簡単にファイルを共有できる⾰新的なソフト「Winny」を開発、試⽤版を「２ちゃんねる」に公開をする。彗星のごとく現れた「Winny」は、本⼈同⼠が直接データのやりとりができるシステムで、瞬く間にシェアを伸ばしていく。しかし、その裏で⼤…" name="description"/><meta content="Winny,映画,感想,動画配信" name="keywords"/><link href="https://filmarks.com/movies/84467" rel="canonical"/><meta content="article" property="og:type"/><meta content="映画『Winny』の感想・レビュー[30014件] | Filmarks" property="og:title"/><meta content="レビュー数：30014件 ／ 平均スコア：★★★★3.8点" property="og:description"/><meta content="https://filmarks.com/movies/84467" property="og:url"/><meta content="https://d2ueuvlup6lbue.cloudfront.net/variants/production/store/fitpad/240/240/nepi4vzdqy0hs4ktq7ynic4zfnow/_.jpg" property="og:image"/><meta content

In [ ]:

# Scrape the review section
# Find the div tag whose class attribute is "p-main-area p-timeline".
# A <div> tag is normally a block-level element that can contain line breaks.
p_main_area = soup.find('div', class_='p-main-area p-timeline')
print(p_main_area)

<div class="p-main-area p-timeline"><div class="p-content-detail-highlight"><h2 class="c2-heading-secondary-l">みんなの反応</h2><div class="p-content-detail-highlight__body"><ul><li class="p-content-detail-highlight__item">Winnyの開発問題を巡る刑事訴訟に関する実話を元にした映画</li><li class="p-content-detail-highlight__item">各登場人物たちの信念や正義のために戦う姿が魅力的</li><li class="p-content-detail-highlight__item">日本の凋落を象徴する特異点的事件</li><li class="p-content-detail-highlight__item">裁判の過程にフォーカスした作品で、実力派揃いの演者も魅せる</li><li class="p-content-detail-highlight__item">金子勇氏の天才的な技術者としての生き様が描かれる</li></ul></div><div class="p-content-detail-highlight__robot"><img alt="" class="" height="20" src="https://d2ueuvlup6lbue.cloudfront.net/assets/pc/component/ico_infor_circle_fill-d00713bed51c21196dfa2419e51ca00de9c6885075c9af5d1540c7d575a2b8d8.svg" width="20"/>この情報はユーザーの感想を元に自動生成しています</div></div><div class="p-content-detail-timeline__header"><h2 class="c2-heading-secondary-l"><div class="c2-heading-secondary-l__lefticon"></div>『Winny』に投稿された感想・評価</h2></di

In [ ]:
# Find div tags whose class attribute is "p-mark__review"
review = p_main_area.find_all('div', class_='p-mark__review')

for i in range(len(review)):  # Iterate through the reviews that were found
    print(review[i])

In [ ]:
# Find div tags whose class attribute is "p-mark__review"
#score = p_main_area.find_all('div', class_='c-rating__score')
score = p_main_area.find_all('div', class_='c2-rating-s__text')
review = p_main_area.find_all('div', class_='p-mark__review')

# Iterate only up to the length of the shorter list to prevent an IndexError
min_length = min(len(score), len(review))
for i in range(min_length):
    print(score[i].text)
    print(review[i].text)

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
from urllib import request

df_list = []
pages = range(100)  # Assume that we will try to retrieve up to 100 pages

for page in pages:
    url = 'https://filmarks.com/movies/84467?page=' + str(page) + ''

    try:
        response = request.urlopen(url)
        soup = BeautifulSoup(response, 'html.parser')  # Explicitly specify 'html.parser' as the parser
        response.close()
    except Exception as e:
        print(f"Error opening URL or parsing page {page}: {e}")
        break  # Stop processing if there is a problem retrieving a page

    # Scrape the review section
    # Find the div tag whose class attribute is "p-main-area p-timeline"
    p_main_area = soup.find('div', class_='p-main-area p-timeline')

    if p_main_area:  # Check whether the main review area was found
        score = p_main_area.find_all('div', class_='c2-rating-s__text')
        review = p_main_area.find_all('div', class_='p-mark__review')

        min_length = min(len(score), len(review))

        if min_length > 0:  # Process only when both scores and reviews are found
            for i in range(min_length):
                _df = pd.DataFrame({'score': [score[i].text],
                                    'review': [review[i].text]})
                df_list.append(_df)
            print(f"page {page} scraped: {min_length} reviews found.")
        else:
            print(f"Page {page}: No valid scores or reviews found within 'p-main-area p-timeline'. Stopping scraping.")
            break  # Stop if no reviews are found on the page, indicating the end of the content
    else:
        print(f"Page {page}: 'p-main-area p-timeline' div not found. Stopping scraping.")
        break  # Stop if the main container cannot be found

# Combine the results into a single DataFrame (only if df_list is not empty)
if df_list:
    df_review = pd.concat(df_list).reset_index(drop=True)
    print(df_review.shape)
    display(df_review.head())  # Use display to make the output easier to view in Colab
else:
    print("No reviews were scraped into df_list.")
    df_review = pd.DataFrame(columns=['score', 'review'])  # Create an empty DataFrame if no data was retrieved

Page 0: No valid scores or reviews found within 'p-main-area p-timeline'. Stopping scraping.
No reviews were scraped into df_list.


In [ ]:
# Replace missing scores, represented by "-", with 0
df_review['score'].replace("-", 0 ,inplace=True)

# Convert scores to numeric values
df_review['score'] = df_review['score'].astype(float)

/tmp/ipykernel_3757/1450300065.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_review['score'].replace("-", 0 ,inplace=True)


In [ ]:
# Save the scraped DataFrame as a CSV file
df_review.to_csv('/content/drive/My Drive/test/review.csv', index=False, encoding='utf_8_sig')

In [ ]:
import pandas as pd
import re

In [ ]:
df = pd.read_csv("/content/drive/My Drive/test/review.csv")

In [ ]:
df['review'] = df['review'].str.replace('□', '')  # Remove symbols
df['review'] = df['review'].str.replace('*', '')
df['review'] = df['review'].str.replace('★', '')
df['review'] = df['review'].str.replace('☆', '')
df['review'] = df['review'].str.replace('😣', '')
df['review'] = df['review'].str.replace('◎', '')
df['review'] = df['review'].str.replace('💻', '')
df['review'] = df['review'].str.replace('🥹', '')
df['review'] = df['review'].str.replace('💢', '')
df['review'] = df['review'].str.replace('😭', '')
df['review'] = df['review'].str.replace('😨', '')
df['review'] = df['review'].str.replace('😶', '')
df['review'] = df['review'].str.replace('✈️', '')
df['review'] = df['review'].str.replace('🥹', '')
df['review'] = df['review'].str.replace('🎞️', '')
df['review'] = df['review'].str.replace('▼', '')
df['review'] = df['review'].str.replace('😱', '')
df['review'] = df['review'].str.replace('✨', '')
df['review'] = df['review'].str.replace('💦', '')
df['review'] = df['review'].str.replace('😊', '')
df['review'] = df['review'].str.replace('👍', '')
df['review'] = df['review'].str.replace('🙏', '')
df['review'] = df['review'].str.replace('🐲', '')
df['review'] = df['review'].str.replace('🐉', '')
df['review'] = df['review'].str.replace('🤩', '')
df['review'] = df['review'].str.replace('⭕️', '')
df['review'] = df['review'].str.replace('🎬', '')
df['review'] = df['review'].str.replace('🫣', '')
df['review'] = df['review'].str.replace('🔪', '')
df['review'] = df['review'].str.replace('😓', '')
df['review'] = df['review'].str.replace('🥲', '')
df['review'] = df['review'].str.replace('⚫︎', '')
df['review'] = df['review'].str.replace('😵‍💫', '')
df['review'] = df['review'].str.replace('🥲', '')
df['review'] = df['review'].str.replace('👏', '')
df['review'] = df['review'].str.replace('⭐️', '')
df['review'] = df['review'].str.replace('◯', '')
df['review'] = df['review'].str.replace('×', '')
df['review'] = df['review'].str.replace('🫶🏻', '')
df['review'] = df['review'].str.replace('☺️', '')

In [ ]:
# Save the data after noise removal as a CSV file
df.to_csv("/content/drive/My Drive/test/review_clean.csv", encoding='utf_8_sig')